# POC 3 - Potencial de mercado de Fiança por região

```

POC 3 - Potencial de mercado de Fiança por região

Pergunta: onde está o mercado de aluguel que a Loft pode endereçar?

Base: painel de anúncios fornecido pela Loft (BigQuery), por mês x UF x
cidade x bairro x operação (venda/aluguel) x tipo de imóvel.

Saídas (reports/tables e reports/figures):
  - qualidade_meses_aluguel.csv      : cobertura mensal e meses descartados
  - potencial_uf.csv                 : potencial por estado
  - potencial_cidades.csv            : potencial por cidade (tabela principal)
  - potencial_distritos.csv          : potencial por distrito (onde existir)
  - cobertura_mensal_aluguel.png     : mostra a quebra de fonte da base
  - top_cidades_potencial.png        : ranking das 20 maiores cidades

Regras de limpeza (justificadas na análise exploratória da base):
  1. Só aluguel (Fiança é garantia locatícia).
  2. Só a partir de nov/2024: antes disso a cobertura é 20 a 50x menor
     (mudança de fonte, não crescimento de mercado).
  3. 'apartment_house' = 'apartment' + 'house'. Usamos só 'apartment_house'
     (residencial) para não contar em dobro. 'other' (comercial/terrenos)
     fica fora do potencial principal e é reportado à parte.
  4. Meses com cobertura anômala (total nacional < 60% da mediana do
     período) são descartados. Ex.: meses incompletos ou falhas de captura.
  5. Preço só é usado se estiver entre R$ 300 e R$ 30.000 e a linha tiver
     pelo menos 5 anúncios (mediana de 1 ou 2 anúncios não é confiável).
  6. As colunas *_yoy_pct da base NÃO são usadas: herdam a quebra de fonte.

Como a cobertura da base oscila mês a mês, as comparações entre regiões
usam PARTICIPAÇÃO no total nacional do mês (share), que é robusta a
variações de cobertura que afetam o país inteiro.
```

## Imports

In [ ]:
import os
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from dotenv import load_dotenv

## Parâmetros

In [ ]:
ARQUIVO_LOFT = "bq-results-20261005-170605-1791220258075.csv"

INICIO_SERIE_CONFIAVEL = "2024-11-01"
JANELA_MESES = 6                 # meses válidos mais recentes usados no potencial
LIMIAR_MES_ANOMALO = 0.60        # mês com total < 60% da mediana é descartado

PRECO_MIN = 300
PRECO_MAX = 30_000
MIN_ANUNCIOS_PRECO = 5

TIPO_RESIDENCIAL = "apartment_house"   # já é apartment + house
TIPO_OUTROS = "other"

# Faixas de prioridade pela participação acumulada no volume nacional
FAIXAS = [(0.50, "A"), (0.80, "B"), (0.95, "C"), (1.01, "D")]

## Caminhos

In [ ]:
# No notebook não existe __file__: o projeto é a pasta acima de notebooks/
PROJECT_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
load_dotenv(PROJECT_DIR / ".env")

caminho_dados = os.getenv("DATA_DIR")

if caminho_dados is None:
    raise ValueError("A variável DATA_DIR não foi encontrada. Confira o arquivo .env.")

DATA_DIR = Path(caminho_dados)
ARQUIVO = DATA_DIR / os.getenv("LOFT_FILE", ARQUIVO_LOFT)

if not ARQUIVO.exists():
    raise FileNotFoundError(f"Arquivo da Loft não encontrado: {ARQUIVO}")

TABLES_DIR = PROJECT_DIR / "reports" / "tables"
FIGURES_DIR = PROJECT_DIR / "reports" / "figures"

TABLES_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

## Leitura

In [ ]:
colunas = [
    "snapshot_month",
    "normalized_state_seed",
    "normalized_city_seed",
    "normalized_neighbourhood_seed",
    "operation_type",
    "property_type_group",
    "n_listings",
    "median_price_mm3",
    "median_area_mm3",
    "distrito",
]

texto = [
    "normalized_state_seed",
    "normalized_city_seed",
    "normalized_neighbourhood_seed",
    "operation_type",
    "property_type_group",
    "distrito",
]

df = pd.read_csv(
    ARQUIVO,
    usecols=colunas,
    dtype={c: str for c in texto},
    parse_dates=["snapshot_month"],
)

df = df.rename(
    columns={
        "snapshot_month": "mes",
        "normalized_state_seed": "uf",
        "normalized_city_seed": "cidade",
        "normalized_neighbourhood_seed": "bairro",
        "operation_type": "operacao",
        "property_type_group": "tipo",
        "n_listings": "anuncios",
        "median_price_mm3": "aluguel_mediano",
        "median_area_mm3": "area_mediana",
    }
)

print("\nBase completa:", df.shape)
print("Período:", df["mes"].min().date(), "a", df["mes"].max().date())

## Checagem: apartment_house = apartment + house

In [ ]:
aluguel_total = df[df["operacao"] == "rent"]

checagem = (
    aluguel_total.pivot_table(
        index=["mes", "cidade", "bairro"],
        columns="tipo",
        values="anuncios",
        aggfunc="sum",
        fill_value=0,
    )
)

soma_ok = (
    checagem["apartment"] + checagem["house"] == checagem["apartment_house"]
).mean()

print(f"\nLinhas em que apartment + house == apartment_house: {soma_ok:.1%}")

## Cobertura mensal e meses anômalos

In [ ]:
residencial_todo = aluguel_total[aluguel_total["tipo"] == TIPO_RESIDENCIAL]

cobertura = (
    residencial_todo.groupby("mes")
    .agg(
        anuncios=("anuncios", "sum"),
        cidades=("cidade", "nunique"),
        bairros=("bairro", "nunique"),
    )
    .reset_index()
)

periodo_confiavel = cobertura["mes"] >= INICIO_SERIE_CONFIAVEL
mediana_periodo = cobertura.loc[periodo_confiavel, "anuncios"].median()

cobertura["razao_mediana"] = cobertura["anuncios"] / mediana_periodo
cobertura["status"] = np.select(
    [
        ~periodo_confiavel,
        cobertura["razao_mediana"] < LIMIAR_MES_ANOMALO,
    ],
    [
        "descartado - antes da quebra de fonte",
        "descartado - cobertura anômala",
    ],
    default="válido",
)

meses_validos = cobertura.loc[cobertura["status"] == "válido", "mes"].sort_values()
meses_janela = meses_validos.iloc[-JANELA_MESES:]

cobertura["na_janela"] = cobertura["mes"].isin(meses_janela)

print("\nCobertura mensal (aluguel residencial):")
print(cobertura.to_string(index=False))

print("\nMeses usados no potencial:")
print([m.strftime("%Y-%m") for m in meses_janela])

## Base de trabalho

In [ ]:
aluguel = aluguel_total[aluguel_total["mes"].isin(meses_janela)].copy()

residencial = aluguel[aluguel["tipo"] == TIPO_RESIDENCIAL].copy()
outros = aluguel[aluguel["tipo"] == TIPO_OUTROS].copy()

residencial["preco_valido"] = (
    residencial["aluguel_mediano"].between(PRECO_MIN, PRECO_MAX)
    & (residencial["anuncios"] >= MIN_ANUNCIOS_PRECO)
)

print("\nLinhas residenciais na janela:", len(residencial))
print(
    "Anúncios com preço válido:",
    f"{residencial.loc[residencial['preco_valido'], 'anuncios'].sum() / residencial['anuncios'].sum():.1%}",
)

## Função: mediana ponderada

In [ ]:
def mediana_ponderada(valores, pesos):
    valores = np.asarray(valores, dtype=float)
    pesos = np.asarray(pesos, dtype=float)

    if len(valores) == 0 or pesos.sum() == 0:
        return np.nan

    ordem = np.argsort(valores)
    valores = valores[ordem]
    acumulado = np.cumsum(pesos[ordem])

    return valores[np.searchsorted(acumulado, acumulado[-1] / 2)]


def aluguel_referencia(base, chaves):
    """Mediana do aluguel ponderada pelo nº de anúncios, por grupo."""
    validos = base[base["preco_valido"]]

    return (
        validos.groupby(chaves)
        .apply(
            lambda g: mediana_ponderada(g["aluguel_mediano"], g["anuncios"]),
            include_groups=False,
        )
        .rename("aluguel_ref")
        .reset_index()
    )

## Preço de referência por cidade/mês (e fallback por uf/mês)

In [ ]:
ref_cidade = aluguel_referencia(residencial, ["mes", "uf", "cidade"])
ref_uf = aluguel_referencia(residencial, ["mes", "uf"]).rename(
    columns={"aluguel_ref": "aluguel_ref_uf"}
)

residencial = residencial.merge(ref_cidade, on=["mes", "uf", "cidade"], how="left")
residencial = residencial.merge(ref_uf, on=["mes", "uf"], how="left")

# Preço usado no cálculo do volume:
# a mediana da própria linha, se confiável; senão, a referência da cidade;
# se a cidade não tiver preço confiável, a referência da UF.
residencial["preco_usado"] = np.where(
    residencial["preco_valido"],
    residencial["aluguel_mediano"],
    residencial["aluguel_ref"].fillna(residencial["aluguel_ref_uf"]),
)

residencial["volume_aluguel"] = residencial["anuncios"] * residencial["preco_usado"]

sem_preco = residencial["preco_usado"].isna()
print(f"\nAnúncios sem nenhum preço de referência: {residencial.loc[sem_preco, 'anuncios'].sum()}")

## Participação nacional por mês

In [ ]:
total_mes = (
    residencial.groupby("mes")
    .agg(anuncios_br=("anuncios", "sum"), volume_br=("volume_aluguel", "sum"))
    .reset_index()
)

residencial = residencial.merge(total_mes, on="mes", how="left")

## Função: potencial por nível geográfico

In [ ]:
meses_recentes = meses_janela.iloc[-3:]
meses_anteriores = meses_janela.iloc[:-3]


def potencial_por(chaves, base=residencial):
    mensal = (
        base.groupby(["mes"] + chaves)
        .agg(
            anuncios=("anuncios", "sum"),
            volume_aluguel=("volume_aluguel", "sum"),
            anuncios_br=("anuncios_br", "first"),
            volume_br=("volume_br", "first"),
            bairros_ativos=("bairro", "nunique"),
        )
        .reset_index()
    )

    mensal["share_anuncios"] = mensal["anuncios"] / mensal["anuncios_br"]
    mensal["share_volume"] = mensal["volume_aluguel"] / mensal["volume_br"]

    resumo = (
        mensal.groupby(chaves)
        .agg(
            meses_presentes=("mes", "nunique"),
            anuncios_mes=("anuncios", "mean"),
            volume_aluguel_mes=("volume_aluguel", "mean"),
            share_anuncios=("share_anuncios", "mean"),
            share_volume=("share_volume", "mean"),
            bairros_ativos=("bairros_ativos", "max"),
        )
        .reset_index()
    )

    # Tendência: share dos 3 meses mais recentes vs. dos meses anteriores.
    # Share (e não volume absoluto) para não confundir com oscilação de cobertura.
    recente = (
        mensal[mensal["mes"].isin(meses_recentes)]
        .groupby(chaves)["share_volume"].mean().rename("share_recente")
    )
    anterior = (
        mensal[mensal["mes"].isin(meses_anteriores)]
        .groupby(chaves)["share_volume"].mean().rename("share_anterior")
    )

    resumo = resumo.merge(recente, on=chaves, how="left")
    resumo = resumo.merge(anterior, on=chaves, how="left")
    resumo["tendencia_share_pct"] = (
        (resumo["share_recente"] / resumo["share_anterior"] - 1) * 100
    )

    # Aluguel de referência da janela inteira (mediana ponderada)
    precos = (
        base[base["preco_valido"]]
        .groupby(chaves)
        .apply(
            lambda g: mediana_ponderada(g["aluguel_mediano"], g["anuncios"]),
            include_groups=False,
        )
        .rename("aluguel_mediano")
        .reset_index()
    )
    area = (
        base[base["preco_valido"]]
        .groupby(chaves)
        .apply(
            lambda g: mediana_ponderada(g["area_mediana"], g["anuncios"]),
            include_groups=False,
        )
        .rename("area_mediana")
        .reset_index()
    )

    resumo = resumo.merge(precos, on=chaves, how="left")
    resumo = resumo.merge(area, on=chaves, how="left")

    # Ranking e faixas de prioridade
    resumo = resumo.sort_values("share_volume", ascending=False).reset_index(drop=True)
    resumo["ranking"] = np.arange(1, len(resumo) + 1)
    resumo["share_acumulado"] = resumo["share_volume"].cumsum()

    limites = [f[0] for f in FAIXAS]
    rotulos = [f[1] for f in FAIXAS]
    resumo["faixa_prioridade"] = pd.cut(
        resumo["share_acumulado"].shift(fill_value=0),
        bins=[-0.01] + limites,
        labels=rotulos,
    )

    # Tendência pouco confiável se a região não apareceu em todos os meses
    resumo.loc[resumo["meses_presentes"] < len(meses_janela), "tendencia_share_pct"] = np.nan

    return resumo

## Potencial por uf

In [ ]:
potencial_uf = potencial_por(["uf"])

print("\nPotencial por UF (top 10):")
print(
    potencial_uf[
        ["ranking", "uf", "anuncios_mes", "aluguel_mediano",
         "share_volume", "tendencia_share_pct", "faixa_prioridade"]
    ].head(10).round(3).to_string(index=False)
)

## Potencial por cidade

In [ ]:
potencial_cidades = potencial_por(["uf", "cidade"])

# Anúncios de imóveis "other" (comerciais, terrenos etc.) só como referência
outros_cidade = (
    outros.groupby(["mes", "uf", "cidade"])["anuncios"].sum()
    .groupby(["uf", "cidade"]).mean()
    .rename("anuncios_outros_mes")
    .reset_index()
)

potencial_cidades = potencial_cidades.merge(
    outros_cidade, on=["uf", "cidade"], how="left"
)

print(f"\nCidades com aluguel residencial na janela: {len(potencial_cidades)}")
print("\nCidades por faixa de prioridade:")
print(potencial_cidades["faixa_prioridade"].value_counts().sort_index())

print("\nTop 20 cidades:")
print(
    potencial_cidades[
        ["ranking", "uf", "cidade", "anuncios_mes", "aluguel_mediano",
         "volume_aluguel_mes", "share_volume", "tendencia_share_pct",
         "faixa_prioridade"]
    ].head(20).round(3).to_string(index=False)
)

## Potencial por distrito

In [ ]:
com_distrito = residencial[residencial["distrito"].notna()].copy()

potencial_distritos = potencial_por(["uf", "cidade", "distrito"], base=com_distrito)

# Participação dentro da própria cidade (mais útil que a nacional aqui)
potencial_distritos["share_na_cidade"] = (
    potencial_distritos["volume_aluguel_mes"]
    / potencial_distritos.groupby(["uf", "cidade"])["volume_aluguel_mes"].transform("sum")
)

print(f"\nDistritos com dados: {len(potencial_distritos)}")
print("Cidades com distrito preenchido:", potencial_distritos["cidade"].nunique())

print("\nTop 15 distritos de São Paulo:")
print(
    potencial_distritos[potencial_distritos["cidade"] == "SAO PAULO"][
        ["distrito", "anuncios_mes", "aluguel_mediano", "share_na_cidade",
         "tendencia_share_pct"]
    ].head(15).round(3).to_string(index=False)
)

## Salvar tabelas

In [ ]:
cobertura.to_csv(TABLES_DIR / "qualidade_meses_aluguel.csv", index=False)
potencial_uf.to_csv(TABLES_DIR / "potencial_uf.csv", index=False)
potencial_cidades.to_csv(TABLES_DIR / "potencial_cidades.csv", index=False)
potencial_distritos.to_csv(TABLES_DIR / "potencial_distritos.csv", index=False)

print("\nTabelas salvas em:", TABLES_DIR)

## Gráfico - cobertura mensal (mostra a quebra de fonte)

In [ ]:
cores = cobertura["status"].map(
    {
        "válido": "#1f6f5c",
        "descartado - antes da quebra de fonte": "#b8b8b8",
        "descartado - cobertura anômala": "#e07a5f",
    }
)

fig, ax = plt.subplots(figsize=(12, 5))
ax.bar(cobertura["mes"], cobertura["anuncios"], width=25, color=cores)

for mes, valor in zip(cobertura["mes"], cobertura["anuncios"]):
    if mes in set(meses_janela):
        ax.text(mes, valor, "●", ha="center", va="bottom", color="#1f6f5c", fontsize=8)

ax.text(
    0.99, 0.97, "● = mês usado no cálculo do potencial",
    transform=ax.transAxes, ha="right", va="top", fontsize=9, color="#1f6f5c",
)

ax.set_title("Anúncios de aluguel residencial por mês - cobertura da base Loft")
ax.set_ylabel("Anúncios")
ax.ticklabel_format(axis="y", style="plain")

from matplotlib.patches import Patch
ax.legend(
    handles=[
        Patch(color="#b8b8b8", label="Antes da quebra de fonte (descartado)"),
        Patch(color="#e07a5f", label="Cobertura anômala (descartado)"),
        Patch(color="#1f6f5c", label="Válido"),
    ],
    loc="upper left",
)

plt.tight_layout()
plt.savefig(FIGURES_DIR / "cobertura_mensal_aluguel.png", dpi=200, bbox_inches="tight")
plt.show()
plt.close()

## Gráfico - top 20 cidades por potencial

In [ ]:
top = potencial_cidades.head(20).iloc[::-1]
cores_faixa = top["faixa_prioridade"].astype(str).map(
    {"A": "#1f6f5c", "B": "#e07a5f", "C": "#f2cc8f", "D": "#b8b8b8"}
)

fig, ax = plt.subplots(figsize=(10, 7))
ax.barh(top["cidade"] + " (" + top["uf"] + ")", top["share_volume"] * 100, color=cores_faixa)

ax.set_title("Participação no volume de aluguel residencial anunciado - top 20 cidades")
ax.set_xlabel("% do volume nacional (R$ de aluguel anunciado)")

ax.legend(
    handles=[
        Patch(color="#1f6f5c", label="Faixa A (50% do volume)"),
        Patch(color="#e07a5f", label="Faixa B (até 80%)"),
        Patch(color="#f2cc8f", label="Faixa C (até 95%)"),
    ],
    loc="lower right",
)

plt.tight_layout()
plt.savefig(FIGURES_DIR / "top_cidades_potencial.png", dpi=200, bbox_inches="tight")
plt.show()
plt.close()

## Conclusão

In [ ]:
faixa_a = potencial_cidades[potencial_cidades["faixa_prioridade"] == "A"]
faixa_ab = potencial_cidades[potencial_cidades["faixa_prioridade"].isin(["A", "B"])]

print("\n============================")
print("CONCLUSÃO - POTENCIAL DE MERCADO")
print("============================")
print(f"Janela analisada: {meses_janela.iloc[0]:%Y-%m} a {meses_janela.iloc[-1]:%Y-%m}")
print(f"Cidades com aluguel residencial: {len(potencial_cidades)}")
print(f"{len(faixa_a)} cidades concentram 50% do volume de aluguel anunciado.")
print(f"{len(faixa_ab)} cidades concentram 80%.")
print(
    "\nO potencial aqui é de OFERTA anunciada (proxy do mercado de locação). "
    "Ele vira o denominador da análise de eficiência quando cruzado com dados "
    "de mídia/demanda (ex.: Google Trends) e com dados públicos de imobiliárias "
    "(CNPJ, CNAE 6821-8/02 e 6822-6/00) e de domicílios alugados (Censo 2022)."
)